In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/filtered_exoplanets.csv")
print(df.shape)
df.head()

(5757, 14)


,pl_name,hostname,pl_rade,pl_bmasse,pl_orbsmax,pl_orbeccen,pl_eqt,pl_insol,st_teff,st_rad,st_mass,st_age,st_spectype,disc_year
0,HD 77946 b,HD 77946,2.896461,10.90000,0.07300,0.211,1128.86,413.1008,6007.0,1.3174,1.1973,3.100,F5,2024.0
1,LHS 1478 b,LHS 1478,1.242000,2.33000,0.01848,NaN,595.00,23.3722,3381.0,0.2460,0.2360,NaN,m3 V,2021.0
2,TOI-5143 c,TOI-5143,16.253022,NaN,0.05602,NaN,904.48,158.1646,5183.0,0.8520,0.8640,7.400,K V,2025.0
3,TOI-815 c,TOI-815,2.620000,23.50000,0.19300,0.220,469.00,8.0300,4869.0,0.7700,0.7760,0.200,K3 V,2024.0
4,WASP-171 b,WASP-171,10.984820,344.52772,0.05040,0.000,1642.00,1200.2400,5965.0,1.6370,1.1710,5.908,G0,2019.0


In [2]:
# Bulk density proxy (Earth-relative units — NOT real g/cm³, just a comparative signal)
df['density_proxy'] = df['pl_bmasse'] / (df['pl_rade'] ** 3)

print(df['density_proxy'].describe())

count    5701.000000
mean        2.530757
std        49.181701
min         0.005487
25%         0.248832
50%         0.474085
75%         0.850209
max      2517.401320
Name: density_proxy, dtype: float64


In [3]:
def get_spectral_class(spectype):
    if pd.isna(spectype):
        return np.nan
    return str(spectype).strip()[0].upper()

df['spectral_class'] = df['st_spectype'].apply(get_spectral_class)
print(df['spectral_class'].value_counts(dropna=False))

spectral_class
NaN    3504
G       755
K       638
M       531
F       288
A        23
B        10
W         2
D         2
L         1
S         1
Y         1
T         1
Name: count, dtype: int64


In [4]:
df['pl_orbeccen_filled'] = df['pl_orbeccen'].fillna(0)
print(f"Eccentricity nulls filled: {df['pl_orbeccen'].isna().sum()}")

Eccentricity nulls filled: 533


In [5]:
# Confirm no unexpected nulls remain in your engineered columns where they shouldn't be
print(df[['density_proxy', 'spectral_class', 'pl_orbeccen_filled']].isna().sum())

# Spot-check a few real planets
print(df[df['pl_name'].str.contains('Kepler-22|TRAPPIST-1 e|51 Eri', na=False)]
        [['pl_name', 'pl_rade', 'pl_bmasse', 'density_proxy', 'spectral_class']])

density_proxy           56
spectral_class        3504
pl_orbeccen_filled       0
dtype: int64
           pl_name  pl_rade    pl_bmasse  density_proxy spectral_class
221   Kepler-222 b     3.16    10.100000       0.320081            NaN
328   Kepler-223 e     4.60     4.800000       0.049314            NaN
430   Kepler-227 c     3.04     9.480000       0.337433            NaN
473   Kepler-224 b     1.39     2.510000       0.934608            NaN
531   Kepler-226 c     2.27    45.200000       3.864211            NaN
744   Kepler-229 d     3.85    14.200000       0.248832            NaN
922   Kepler-225 c     1.84     4.040000       0.648527              M
1018   Kepler-22 b     2.10     9.100000       0.982615              G
1149  Kepler-229 c     4.92    21.500000       0.180527            NaN
1179  Kepler-221 b     1.71     5.955750       1.191100            NaN
1564  Kepler-228 d     4.04    15.400000       0.233548            NaN
1613  Kepler-222 d     3.69    13.200000       0.26272

In [6]:
import os
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/clean_exoplanets.csv", index=False)
print(df.shape)

(5757, 17)


## Session 3 summary

- Engineered `density_proxy` = pl_bmasse / pl_rade³ (Earth-relative bulk density signal)
- Engineered `spectral_class` from `st_spectype` (first letter of spectral type, used 
  for the stellar stability factor)
- Created `pl_orbeccen_filled` (nulls set to 0, following the convention that 
  unmeasured eccentricity typically indicates a well-fit circular orbit) — original 
  `pl_orbeccen` column preserved unchanged for transparency
- Output: 5,757 planets, ready for scoring (Phase 2)